# day-20-agent-frameworks — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [7]:
# ---- Solution 1 ----
class StreamRunnable(Runnable):
    def stream(self, x):
        out = self.invoke(x)
        for tok in re.findall(r"\S+\s*", str(out)):
            yield tok

sc = StreamRunnable((PromptTemplate("Echo: {t}") | FakeLLM() | StrOutputParser()).fn)
print("".join(t for t in sc.stream({"t": "hello world from a stream"})))

[LLM answer to: Echo: hello world from a stream...]


In [8]:
# ---- Solution 2 ----
def RunnableBranch(*pairs, default):
    def run(x):
        for cond, chain in pairs:
            if cond(x): return chain.invoke(x)
        return default.invoke(x)
    return Runnable(run)

sentiment_chain = PromptTemplate("Classify the sentiment of: {text}") | FakeLLM() | StrOutputParser()
generic_chain   = PromptTemplate("Answer: {text}") | FakeLLM() | StrOutputParser()
router = RunnableBranch(
    (lambda d: "feel" in d["text"] or "sentiment" in d["text"], sentiment_chain),
    default=generic_chain)
print(router.invoke({"text": "sentiment: I love it"}))
print(router.invoke({"text": "what time is it"}))

positive
[LLM answer to: Answer: what time is it...]


In [9]:
# ---- Solution 3 ----
def as_query_engine_filtered(index, source=None, top_k=2):
    idxs = [i for i, n in enumerate(index.nodes) if source is None or n.metadata.get("source") == source]
    sub_nodes = [index.nodes[i] for i in idxs]; sub_vecs = index.vecs[idxs]
    return QueryEngine(sub_nodes, sub_vecs, lambda p: f"[from {source}] " + p[:40], top_k=top_k)

qe_api = as_query_engine_filtered(index, source="api-docs")
r = qe_api.query("rate limit")
print("answer:", r.response)
print("all sources api-docs?", all(n.metadata["source"] == "api-docs" for n in r.source_nodes))

answer: [from api-docs] Context:
- The REST API allows 600 reque
all sources api-docs? True


### Solutions 4, 5, 6 (sketch)

**S4:** LCEL: `RunnableParallel(context=retriever, question=passthrough) | prompt | llm |
parser` — one expression. LlamaIndex: `VectorStoreIndex.from_documents(docs).as_query_engine()`
— two calls. LlamaIndex wins on line count *for the standard RAG path*; LCEL wins when you
need custom steps (reranking, gating, multi-tool) between retrieve and generate.

**S5:** tree-summarize on N nodes: summarize pairs → ⌈N/2⌉ summaries → repeat → 1 root. Token
cost ≈ `2·total_node_tokens` (each node read ~twice) vs stuffing (`1·total` but a bigger single
prompt and lost-in-the-middle risk, Day 04). Worth it when N is large or nodes are long.

**S6:** the four packages pull ~40–80 transitive deps (pydantic, tenacity, SQLAlchemy,
dataclasses-json, tiktoken, numpy, aiohttp, orjson, typing-extensions pins, plus each
integration's own client). Justified when you use ≥ 3 non-trivial integrations or need
LangSmith/LangGraph; not justified for a single-model, single-vector-store app you could write
in 150 lines (Weeks 5–7).

### Answer key
1. A component with an `.invoke()` (and `stream`/`batch`/`ainvoke`); `|` composes two
   Runnables so the output of the left feeds the input of the right — `prompt | llm | parser`.
2. `VectorStoreIndex.from_documents(docs)` (hides chunking + embedding + vector storage) and
   `index.as_query_engine().query(q)` (hides retrieval + prompt construction + answer
   synthesis + source attribution).
3. LlamaIndex is retrieval-first (indexing, query engines, connectors); LangChain is
   orchestration-first (chains, agents, integrations), with LangGraph for stateful agents.
4. For anything beyond a simple tool loop — branching control flow, explicit state, human-in-
   the-loop checkpoints, durable/resumable execution.
5. A churning dependency tree; an abstraction layer you must learn and debug through (and that
   can fight non-standard use cases). Also: version-pinning friction and slower cold starts.
6. LlamaIndex — the standard RAG-with-citations path is `from_documents` + `as_query_engine`,
   it has the deepest retrieval machinery and the most data connectors, and vector stores are
   swappable via one integration import. LangChain would work but you'd assemble more of the
   retrieval/synthesis yourself.
7. So ground truth and metrics survive a framework swap; the eval measures *your task*, not
   the framework, and you want to be able to migrate (or drop the framework) without losing
   the ability to tell whether quality changed.